In [0]:
-- Drop and create vehicle dimension table
DROP TABLE IF EXISTS practice.bricks.dim_vehicle;

CREATE TABLE practice.bricks.dim_vehicle (
  vehicle_id BIGINT GENERATED ALWAYS AS IDENTITY,
  vin STRING,
  year INT,
  make STRING,
  model STRING,
  trim STRING,
  body_style STRING,
  transmission STRING,
  color STRING,
  interior STRING,
  CONSTRAINT pk_vehicle PRIMARY KEY (vehicle_id)
);

-- Populate vehicle dimension table
INSERT INTO practice.bricks.dim_vehicle (vin, year, make, model, trim, body_style, transmission, color, interior)
SELECT DISTINCT 
  vin,
  year,
  make,
  model,
  trim,
  body AS body_style,
  transmission,
  color,
  interior
FROM practice.bricks.car_sales_data_deduped
WHERE vin IS NOT NULL;

num_affected_rows,num_inserted_rows
550284,550284


In [0]:
-- Drop and create date dimension table
DROP TABLE IF EXISTS practice.bricks.dim_date;

CREATE TABLE practice.bricks.dim_date (
  date_id BIGINT GENERATED ALWAYS AS IDENTITY,
  sale_date DATE,
  year INT,
  month INT,
  month_name STRING,
  day INT,
  weekday STRING,
  CONSTRAINT pk_date PRIMARY KEY (date_id)
);

-- Populate date dimension table
INSERT INTO practice.bricks.dim_date (sale_date, year, month, month_name, day, weekday)
SELECT DISTINCT
  saledate AS sale_date,
  YEAR(saledate) AS year,
  MONTH(saledate) AS month,
  DATE_FORMAT(saledate, 'MMMM') AS month_name,
  DAY(saledate) AS day,
  DATE_FORMAT(saledate, 'EEEE') AS weekday
FROM practice.bricks.car_sales_data_deduped
WHERE saledate IS NOT NULL;

num_affected_rows,num_inserted_rows
171,171


In [0]:
-- Drop and create location dimension table
DROP TABLE IF EXISTS practice.bricks.dim_location;

CREATE TABLE practice.bricks.dim_location (
  location_id BIGINT GENERATED ALWAYS AS IDENTITY,
  state STRING,
  CONSTRAINT pk_location PRIMARY KEY (location_id)
);

-- Populate location dimension table
INSERT INTO practice.bricks.dim_location (state)
SELECT DISTINCT state
FROM practice.bricks.car_sales_data_deduped
WHERE state IS NOT NULL;

num_affected_rows,num_inserted_rows
38,38


In [0]:
-- Drop and create seller dimension table
DROP TABLE IF EXISTS practice.bricks.dim_seller;

CREATE TABLE practice.bricks.dim_seller (
  seller_id BIGINT GENERATED ALWAYS AS IDENTITY,
  seller_name STRING,
  CONSTRAINT pk_seller PRIMARY KEY (seller_id)
);

-- Populate seller dimension table
INSERT INTO practice.bricks.dim_seller (seller_name)
SELECT DISTINCT seller AS seller_name
FROM practice.bricks.car_sales_data_deduped
WHERE seller IS NOT NULL;

num_affected_rows,num_inserted_rows
14190,14190


In [0]:
-- Drop and create fact table
DROP TABLE IF EXISTS practice.bricks.fact_sales;

CREATE TABLE practice.bricks.fact_sales (
  sales_id BIGINT GENERATED ALWAYS AS IDENTITY,
  vehicle_id BIGINT,
  seller_id BIGINT,
  location_id BIGINT,
  date_id BIGINT,
  selling_price DECIMAL(10, 2),
  mmr DECIMAL(10, 2),
  odometer INT,
  condition STRING,
  CONSTRAINT pk_sales PRIMARY KEY (sales_id),
  CONSTRAINT fk_vehicle FOREIGN KEY (vehicle_id) REFERENCES practice.bricks.dim_vehicle(vehicle_id),
  CONSTRAINT fk_seller FOREIGN KEY (seller_id) REFERENCES practice.bricks.dim_seller(seller_id),
  CONSTRAINT fk_location FOREIGN KEY (location_id) REFERENCES practice.bricks.dim_location(location_id),
  CONSTRAINT fk_date FOREIGN KEY (date_id) REFERENCES practice.bricks.dim_date(date_id)
);

-- Populate fact table with joins
INSERT INTO practice.bricks.fact_sales (vehicle_id, seller_id, location_id, date_id, selling_price, mmr, odometer, condition)
SELECT 
  v.vehicle_id,
  sl.seller_id,
  l.location_id,
  d.date_id,
  s.sellingprice AS selling_price,
  s.mmr,
  s.odometer,
  CAST(s.condition AS STRING) AS condition
FROM practice.bricks.car_sales_data_deduped s
INNER JOIN practice.bricks.dim_vehicle v 
  ON s.vin = v.vin
INNER JOIN practice.bricks.dim_seller sl 
  ON s.seller = sl.seller_name
INNER JOIN practice.bricks.dim_location l 
  ON s.state = l.state
INNER JOIN practice.bricks.dim_date d 
  ON s.saledate = d.sale_date
WHERE s.vin IS NOT NULL 
  AND s.seller IS NOT NULL 
  AND s.state IS NOT NULL 
  AND s.saledate IS NOT NULL;

num_affected_rows,num_inserted_rows
550284,550284


In [0]:
-- Drop and create analytical view table
DROP TABLE IF EXISTS practice.bricks.car_sales_analytical_view;

CREATE TABLE practice.bricks.car_sales_analytical_view AS
SELECT 
  -- Fact table columns
  f.sales_id,
  f.selling_price,
  f.mmr,
  f.odometer,
  f.condition,
  
  -- Date dimension columns
  d.sale_date,
  d.year AS sale_year,
  d.month AS sale_month,
  d.month_name AS sale_month_name,
  d.day AS sale_day,
  d.weekday AS sale_weekday,
  
  -- Vehicle dimension columns
  v.vin,
  v.year AS vehicle_year,
  v.make,
  v.model,
  v.trim,
  v.body_style,
  v.transmission,
  v.color,
  v.interior,
  
  -- Seller dimension columns
  s.seller_name,
  
  -- Location dimension columns
  l.state
  
FROM practice.bricks.fact_sales f
INNER JOIN practice.bricks.dim_vehicle v ON f.vehicle_id = v.vehicle_id
INNER JOIN practice.bricks.dim_seller s ON f.seller_id = s.seller_id
INNER JOIN practice.bricks.dim_location l ON f.location_id = l.location_id
INNER JOIN practice.bricks.dim_date d ON f.date_id = d.date_id;

num_affected_rows,num_inserted_rows


In [0]:
-- Query the analytical view - no joins needed!
SELECT 
  sales_id,
  sale_date,
  sale_year,
  sale_month_name,
  vehicle_year,
  make,
  model,
  color,
  seller_name,
  state,
  selling_price,
  mmr,
  odometer,
  condition
FROM practice.bricks.car_sales_analytical_view
LIMIT 10;

sales_id,sale_date,sale_year,sale_month_name,vehicle_year,make,model,color,seller_name,state,selling_price,mmr,odometer,condition
65,2015-06-04,2015,June,2002,Acura,TL,white,automobile acceptance corp,ga,1500.00,2175.00,162190,23
67,2014-12-23,2014,December,2003,Acura,TL,silver,courtesy kia of brandon,fl,1400.00,1350.00,211287,19
80,2015-05-13,2015,May,2003,Acura,TL,gray,classic honda,pa,1900.00,3200.00,134064,19
158,2015-02-25,2015,February,2003,Acura,TL,gray,acura of concord,ca,5000.00,3125.00,111804,25
169,2015-01-13,2015,January,2001,Acura,TL,beige,capital auto auction,tx,2400.00,2800.00,104875,2
185,2014-12-23,2014,December,2001,Acura,TL,green,coachworks,oh,500.00,425.00,272898,0
201,2015-01-21,2015,January,2002,Acura,TL,black,mercedes benz of northlake,nc,5800.00,3225.00,100407,35
202,2015-05-21,2015,May,2002,Acura,TL,silver,automobile acceptance corp,ga,2700.00,3075.00,130369,19
203,2015-02-27,2015,February,2002,Acura,TL,white,showroom auto llc,pa,2700.00,3125.00,124414,19
223,2015-01-07,2015,January,2003,Acura,TL,black,car financial,ca,1500.00,2300.00,169168,19


In [0]:
SELECT 
  f.sales_id,
  d.sale_date,
  d.year,
  d.month_name,
  v.year AS vehicle_year,
  v.make,
  v.model,
  v.color,
  s.seller_name,
  l.state,
  f.selling_price,
  f.mmr,
  f.odometer,
  f.condition
FROM practice.bricks.fact_sales f
INNER JOIN practice.bricks.dim_vehicle v ON f.vehicle_id = v.vehicle_id
INNER JOIN practice.bricks.dim_seller s ON f.seller_id = s.seller_id
INNER JOIN practice.bricks.dim_location l ON f.location_id = l.location_id
INNER JOIN practice.bricks.dim_date d ON f.date_id = d.date_id
LIMIT 10;

sales_id,sale_date,year,month_name,vehicle_year,make,model,color,seller_name,state,selling_price,mmr,odometer,condition
1,2015-02-23,2015,February,2002,Unknown,Unknown,white,performance auto center inc,nc,36000.00,47000.00,79808,25
2,2015-05-28,2015,May,2001,HUMMER,H1,blue,auto city sales/leasing,ca,45750.00,43400.00,65612,21
3,2015-02-05,2015,February,2000,Unknown,Unknown,silver,aaero sweet company,ca,42000.00,40300.00,84028,23
4,2015-02-11,2015,February,1999,Acura,TL,black,enterprise vehicle exchange / tra / rental / tulsa,ga,1050.00,1075.00,233154,19
5,2015-02-05,2015,February,1999,Acura,TL,blue,coggin nissan,fl,2100.00,1325.00,175516,25
6,2014-12-18,2014,December,1999,Acura,TL,red,bargain wheels llc,oh,1600.00,1825.00,113958,0
7,2015-03-05,2015,March,1999,Acura,TL,green,purple heart,va,2800.00,1975.00,130002,2
8,2015-06-05,2015,June,1999,Acura,TL,green,r hollenshead auto sales inc,pa,1450.00,1500.00,153673,19
9,2015-06-04,2015,June,1999,Acura,TL,gray,wayne auto sales inc,nj,2000.00,1725.00,121985,35
10,2014-12-18,2014,December,1999,Acura,TL,silver,elder ford of tampa,fl,1300.00,1875.00,107468,0
